In [1]:
!pip install -U "jax[cpu]"

In [2]:
!pip install equinox

In [4]:
!pip install optax

   ---------------------------------------- 0.0/223.7 kB ? eta -:--:--
   - -------------------------------------- 10.2/223.7 kB ? eta -:--:--
   -------------------------------------- - 215.0/223.7 kB 4.4 MB/s eta 0:00:01
   ---------------------------------------- 223.7/223.7 kB 3.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/133.7 kB ? eta -:--:--
   ---------------------------------------- 133.7/133.7 kB 7.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/98.2 kB ? eta -:--:--
   ---------------------------------------- 98.2/98.2 kB 5.5 MB/s eta 0:00:00


In [65]:
import equinox as eqx # Library for everything jax doesn't have, like neural networks
import jax
import jax.numpy as jnp
import jax.random as jp
import optax # Provides optimizers

In [43]:
# Define the Lorenz system equations
def lorenz(x, y, z, sigma=10, rho=28, beta=8/3):
    dx_dt = sigma * (y - x)
    dy_dt = x * (rho - z) - y
    dz_dt = x * y - beta * z
    return dx_dt, dy_dt, dz_dt

In [75]:
# Runge-Kutta integration method to solve the Lorenz system with noise. The method adds a midpoint between discrete points so more training data
def runge_kutta_lorenz_with_noise(key, x0, y0, z0, sigma=10, rho=28, beta=8/3, dt=0.01, timesteps=10000, noise_std=0.1):
    x_traj, y_traj, z_traj = [x0], [y0], [z0]
    keys = jp.split(key, timesteps)
    
    for i in range(timesteps):
        # Calculate derivatives using the Lorenz equations
        dx1, dy1, dz1 = lorenz(x_traj[-1], y_traj[-1], z_traj[-1], sigma, rho, beta)
        dx2, dy2, dz2 = lorenz(x_traj[-1] + 0.5 * dt * dx1, y_traj[-1] + 0.5 * dt * dy1, z_traj[-1] + 0.5 * dt * dz1, sigma, rho, beta)
        dx3, dy3, dz3 = lorenz(x_traj[-1] + 0.5 * dt * dx2, y_traj[-1] + 0.5 * dt * dy2, z_traj[-1] + 0.5 * dt * dz2, sigma, rho, beta)
        dx4, dy4, dz4 = lorenz(x_traj[-1] + dt * dx3, y_traj[-1] + dt * dy3, z_traj[-1] + dt * dz3, sigma, rho, beta)
        
        # Add noise to the trajectory --> more realisitc, especially observational data
        noise_x = jp.normal(keys[i], ()) * noise_std
        noise_y = jp.normal(keys[i], ()) * noise_std
        noise_z = jp.normal(keys[i], ()) * noise_std
        
        # Update trajectory using Runge-Kutta method
        x_traj.append(x_traj[-1] + (dt / 6) * (dx1 + 2*dx2 + 2*dx3 + dx4) + noise_x)
        y_traj.append(y_traj[-1] + (dt / 6) * (dy1 + 2*dy2 + 2*dy3 + dy4) + noise_y)
        z_traj.append(z_traj[-1] + (dt / 6) * (dz1 + 2*dz2 + 2*dz3 + dz4) + noise_z)
        
    return jnp.array(x_traj), jnp.array(y_traj), jnp.array(z_traj)

In [77]:
# Generate training data using Runge-Kutta with noise
x0, y0, z0 = 1.0, 1.0, 1.0
x_train, y_train, z_train = runge_kutta_lorenz_with_noise(jp.PRNGKey(0), x0, y0, z0)

# Convert training data to torch tensors
x_phys = jnp.column_stack((x_train, y_train, z_train))
print(f"x_phys {x_phys.shape}") # [num of points/time steps, 3] --> ideal input for pytorch, make sense when doing matrix multiplication
print(f"x_phys {type(x_phys)}")

x_phys (10001, 3)
x_phys <class 'jaxlib.xla_extension.ArrayImpl'>


In [78]:
class PINN(eqx.Module):
    layer1: eqx.nn.Linear # This is needed bc can't assign to attributes to layers
    layer2: eqx.nn.Linear # unless defined previously
    layer3: eqx.nn.Linear
    
    def __init__(self, key):
        key1, key2, key3 = jax.random.split(key, 3) # Used to keep track of random state?
        self.layer1 = eqx.nn.Linear(3, 50, key=key1)
        self.layer2 = eqx.nn.Linear(50, 50, key=key2)
        self.layer3 = eqx.nn.Linear(50, 3, key=key3)

    def __call__(self, x):
        x = jax.nn.tanh(self.layer1(x))
        x = jax.nn.tanh(self.layer2(x))
        x = self.layer3(x)
        return x

In [79]:
# Define the loss function
def loss_fn(model, x_phys):
    x_pred = model(x_phys)
    dx_dt_pred, dy_dt_pred, dz_dt_pred = jnp.split(x_pred, 3, axis=1)
    x, y, z = jnp.split(x_phys, 3, axis=1)
    dx_dt, dy_dt, dz_dt = lorenz(x, y, z)
    loss_eqn = jnp.mean((dx_dt_pred - dx_dt) ** 2 + (dy_dt_pred - dy_dt) ** 2 + (dz_dt_pred - dz_dt) ** 2)
    return loss_eqn

In [80]:
@jax.jit # Optimize the function to make it more efficient
def train_step(model, opt_state):
    loss, grads = jax.value_and_grad(loss_fn)(model, x_phys)
    updates, new_state = optimizer.update(grads, opt_state, model)
    new_model = eqx.apply_updates(model, updates)
    return new_model, new_state, loss

In [81]:
# Training function
def train(model, optimizer, opt_state, num_epochs=15000):
    for epoch in range(num_epochs):
        model, opt_state, loss = train_step(model, opt_state)
        if epoch == 1 or (epoch % 100 == 0 and epoch != 0):
            print(f'Epoch [{epoch}/{num_epochs}], Loss: {loss.item()}')

In [82]:
model = PINN(key=jax.random.PRNGKey(0))
optimizer = optax.adam(0.0008)
opt_state = optimizer.init(model)

train(model, optimizer, opt_state)

TypeError: dot_general requires contracting dimensions to have the same shape, got (3,) and (10001,).

In [ ]:
# Training function
def train(model, optimizer, state, epoch):
    loss, grad = eqx.filter_value_and_grad(loss_fn)(model)
    updates, new_state = optimizer.update(grad, state, model)
    new_network = eqx.apply_updates(model, updates)
          
    if epoch == 1 or epoch % 100 == 0 and epoch != 0:
        print(f'Epoch [{epoch}/{num_epochs}], Loss: {loss.item()}')

    return new_network, new_state

In [ ]:
model = PINN(key=jax.random.PRNGKey(0))
optimizer = optax.adam(0.0008)

for epoch in range(15000):
    model = train(model, optimizer, epoch)